Extract all main campus zip codes and Fall 2023 enrollment data for public and private non-profit 4+ year universities.

In [6]:
import pyodbc
import pandas as pd
from pathlib import Path

# Paths

DB_PATH = Path("IPEDS_23-24/IPEDS202324.accdb")
OUT_CSV = Path("university_enrollment_zipcode.csv")

# Helpers

def format_ipeds_zip(value):
    """
    IPEDS ZIP fields may be stored without a dash.
    Examples:
      60637      -> 60637
      060102301  -> 06010-2301
    Returns a normalized string or None.
    """
    if pd.isna(value):
        return None

    s = str(value).strip()

    # remove trailing .0 if Access/pandas coerces to float
    if s.endswith(".0"):
        s = s[:-2]

    # keep digits only
    s = "".join(ch for ch in s if ch.isdigit())

    if not s:
        return None

    # 5-digit ZIP
    if len(s) <= 5:
        return s.zfill(5)

    # 9-digit ZIP+4
    if len(s) == 9:
        return f"{s[:5]}-{s[5:]}"

    # anything else: preserve raw digits
    return s


def get_access_connection(db_path: Path):
    conn_str = (
        r"DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};"
        f"DBQ={db_path};"
    )
    return pyodbc.connect(conn_str)


# Main

def main():
    if not DB_PATH.exists():
        raise FileNotFoundError(f"Database not found: {DB_PATH}")

    conn = get_access_connection(DB_PATH)

    try:
        # EF2023 has multiple records per institution.
        # EFLEVEL = 10 corresponds to "All students total".
        query = """
        SELECT
            h.UNITID,
            h.INSTNM,
            h.ZIP,
            e.EFTOTAL
        FROM
            HD2023 AS h
            INNER JOIN EF2023 AS e
                ON h.UNITID = e.UNITID
        WHERE
            e.EFLEVEL = 10
            AND h.SECTOR IN (1, 2)
        """

        df = pd.read_sql(query, conn)

    finally:
        conn.close()

    # clean up columns
    df = df.rename(columns={
        "INSTNM": "institution_name",
        "ZIP": "zip_raw",
        "EFTOTAL": "fall_2023_total_enrollment"
    })

    # normalize ZIP
    df["zip_code"] = df["zip_raw"].apply(format_ipeds_zip)

    # optional split fields
    df["zip5"] = df["zip_code"].str.extract(r"^(\d{5})", expand=False)
    df["zip4"] = df["zip_code"].str.extract(r"^\d{5}-(\d{4})$", expand=False)

    # tidy final column order
    df = df[
        [
            "UNITID",
            "institution_name",
            "fall_2023_total_enrollment",
            "zip_code",
            "zip5",
            "zip4",
            "zip_raw",
        ]
    ].sort_values(["institution_name", "UNITID"]).reset_index(drop=True)

    df.to_csv(OUT_CSV, index=False)

    print(f"Saved {len(df):,} rows to: {OUT_CSV.resolve()}")
    print("\nPreview:")
    print(df.head(20).to_string(index=False))


if __name__ == "__main__":
    main()

C:\Users\jwram\AppData\Local\Temp\ipykernel_15912\2018138191.py:81: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Saved 2,427 rows to: C:\Users\jwram\.ipython\university_caffeine_analysis\university_enrollment_zipcode.csv

Preview:
 UNITID                                                              institution_name  fall_2023_total_enrollment   zip_code  zip5 zip4    zip_raw
 177834                                       A T Still University of Health Sciences                        3629      63501 63501  NaN      63501
 180203                                                        Aaniiih Nakoda College                         133      59526 59526  NaN      59526
 222178                                                  Abilene Christian University                        5114      79699 79699  NaN      79699
 497037                             Abilene Christian University-Undergraduate Online                         918      75001 75001  NaN      75001
 138558                                          Abraham Baldwin Agricultural College                        3768 31793-2601 31793 2601 31793-2601
